# 07. PRISM 4 km Monthly Climate Download & California Cropping

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/flowracle/notebooks/data_preprocessing/07_download_prism_monthly.ipynb)

This notebook downloads public 4 km monthly gridded precipitation (`ppt`) and
air temperature (`tmean`, `tmax`, `tmin`) GeoTIFF archives from the Oregon
State University PRISM Climate Group web service (`https://services.nacse.org/prism/data/get/us/4km`),
clips them to the extended California snow-fed domain (`32.5–45.5°N`,
`124.0–115.0°W`), and consolidates them into Zarr:
- **Output path**: `${FLOWRACLE_DATA_DIR}/prism-monthly-ca-region/prism_monthly.zarr`

##### Copyright 2026 The Google Research Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

---
*Disclaimer: This is not an officially supported Google product.*

In [ ]:
!pip install -q "numpy>=1.26.0" "pandas>=2.0.0" "requests>=2.31.0" "beautifulsoup4>=4.12.0" "lxml>=4.9.0" "xarray>=2023.12.0" "zarr>=2.16.0,<3.0.0" "dask>=2023.12.0" "tqdm>=4.66.0" "rioxarray>=0.15.0" "rasterio>=1.3.0"

In [ ]:
import datetime
import io
import os
import shutil
import tempfile
import zipfile

import pandas as pd
import requests
import rioxarray
from tqdm import tqdm
import xarray as xr

FLOWRACLE_DATA_DIR = os.environ.get("FLOWRACLE_DATA_DIR", "./flowracle_data")
OUTPUT_DIR = os.path.join(FLOWRACLE_DATA_DIR, "prism-monthly-ca-region")
RAW_ZIP_DIR = os.path.join(OUTPUT_DIR, "raw_zips")
os.makedirs(RAW_ZIP_DIR, exist_ok=True)

BASE_URL = "https://services.nacse.org/prism/data/get/us/4km"
VARIABLES = ["ppt", "tmean", "tmax", "tmin"]
START_YEAR = 1979
NOW = datetime.datetime.now()
END_YEAR = NOW.year
LAST_COMPLETED_MONTH = NOW.replace(day=1) - pd.Timedelta(days=1)

LAT_MIN, LAT_MAX = 32.5, 45.5
LON_MIN, LON_MAX = -124.0, -115.0

## 1. Download Monthly PRISM 4 km ZIP Archives (Resumable)

In [ ]:
def download_prism_month(out_dir, var, year, month):
  zip_path = os.path.join(out_dir, f"prism_{var}_{year}{month:02d}.zip")
  if os.path.exists(zip_path) and os.path.getsize(zip_path) > 0:
    return zip_path
  url = f"{BASE_URL}/{var}/{year}{month:02d}"
  try:
    resp = requests.get(url, timeout=45)
    resp.raise_for_status()
    ctype = resp.headers.get("Content-Type", "")
    if "application/zip" in ctype or not ctype or resp.content[:2] == b"PK":
      with open(zip_path, "wb") as f:
        f.write(resp.content)
      return zip_path
  except requests.RequestException:
    return None
  return None


for var in VARIABLES:
  print(f"Downloading PRISM 4 km variable: {var}")
  for year in tqdm(range(START_YEAR, END_YEAR + 1), desc=var):
    for month in range(1, 13):
      if pd.Timestamp(year=year, month=month, day=1) > LAST_COMPLETED_MONTH:
        continue
      download_prism_month(RAW_ZIP_DIR, var, year, month)

## 2. Crop GeoTIFFs to California Domain & Consolidate into Zarr

In [ ]:
def aggregate_prism_to_zarr(zip_dir, zarr_out_path):
  ds_vars = {}
  with tempfile.TemporaryDirectory() as tmp_dir:
    for var in VARIABLES:
      das, times = [], []
      for year in tqdm(range(START_YEAR, END_YEAR + 1), desc=f"Cropping {var}"):
        for month in range(1, 13):
          zip_path = os.path.join(zip_dir, f"prism_{var}_{year}{month:02d}.zip")
          if not os.path.exists(zip_path):
            continue
          with open(zip_path, "rb") as f:
            raw_bytes = f.read()
          with zipfile.ZipFile(io.BytesIO(raw_bytes)) as zf:
            tifs = [n for n in zf.namelist() if n.endswith(".tif")]
            if not tifs:
              continue
            zf.extract(tifs[0], tmp_dir)
            tif_path = os.path.join(tmp_dir, tifs[0])
            with rioxarray.open_rasterio(tif_path) as raw_da:
              da = raw_da.squeeze("band", drop=True)
              cropped = da.rio.clip_box(
                  minx=LON_MIN, miny=LAT_MIN, maxx=LON_MAX, maxy=LAT_MAX
              ).load()
            das.append(cropped)
            times.append(pd.Timestamp(year=year, month=month, day=1))
            os.remove(tif_path)
      if das:
        ds_vars[var] = xr.concat(das, dim="time").assign_coords(time=times)

  ds = xr.Dataset(ds_vars)
  if os.path.exists(zarr_out_path):
    shutil.rmtree(zarr_out_path)
  ds.to_zarr(zarr_out_path, mode="w")
  print(f"Wrote Zarr store: {zarr_out_path}")
  return ds


prism_zarr_path = os.path.join(OUTPUT_DIR, "prism_monthly.zarr")
prism_ds = aggregate_prism_to_zarr(RAW_ZIP_DIR, prism_zarr_path)
print(prism_ds)